In [39]:
import os
import sys
import pandas as pd
import geopandas as gpd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint
from sklearn.preprocessing import LabelEncoder

from sklearn.tree import export_graphviz
from IPython.display import Image
import graphviz

sys.path.append("../utils")

import config

pd.set_option("display.max_columns", None)


In [40]:
data = pd.read_csv("/capstone/wildfire_prep/data/PUZZLE_PIECES/assembled_puzzle.csv")

In [41]:
data.columns

Index(['inspection_id', 'X_0', 'X_1', 'X_2', 'X_3', 'X_4', 'X_5', 'X_6', 'X_7',
       'X_8',
       ...
       'X_3994', 'X_3995', 'X_3996', 'X_3997', 'X_3998', 'X_3999',
       'basemap_id', 'maj_landcover_code', 'status', 'structure_code'],
      dtype='object', length=4005)

In [42]:
# drop basemap_id 

data = data.drop(columns = 'basemap_id')

In [43]:
data.columns

Index(['inspection_id', 'X_0', 'X_1', 'X_2', 'X_3', 'X_4', 'X_5', 'X_6', 'X_7',
       'X_8',
       ...
       'X_3993', 'X_3994', 'X_3995', 'X_3996', 'X_3997', 'X_3998', 'X_3999',
       'maj_landcover_code', 'status', 'structure_code'],
      dtype='object', length=4004)

## Full Dataset Approach

In [45]:
X = data.drop("status", axis=1)
y = data["status"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)


## Balanced Approach

In [44]:
# Balanced dataset -- 25% non-compliant

compliant = data[data.status == 0]
non_compliant = data[data.status == 1]

# Making non-compliant 25% of the dataset
n_majority_desired = len(non_compliant) * 3

# Downsample number of compliant rows
compliant_down = compliant.sample(n=n_majority_desired, random_state=42)

# reassemble and shuffle
balanced = (
    pd.concat([non_compliant, compliant_down])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

balanced.status.value_counts(normalize=True)

# variable for this downsampling approach is `balanced`

status
0    0.75
1    0.25
Name: proportion, dtype: float64

In [ ]:
# Balanced

X_balanced = balanced.drop("status", axis=1)
y_balanced = balanced["status"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [38]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


KeyboardInterrupt: 

In [20]:
# param_dist = {
#     "n_estimators": randint(50, 500),
#     "max_depth": randint(1, 20),
#     "class_weight": [None, "balanced", "balanced_subsample"],
# }

# rf = RandomForestClassifier(random_state=42)

# rand_search = RandomizedSearchCV(
#     rf,
#     param_distributions=param_dist,
#     n_iter=20,
#     cv=5,
#     scoring="precision",
#     refit=True,
# )

# rand_search.fit(X_train, y_train)

# print("Best precision: ", rand_search.best_score_)
# print("Best params:    ", rand_search.best_params_)


In [ ]:
# Create a variable for the best model
best_rf = rand_search.best_estimator_

# Print the best hyperparameters
print("Best hyperparameters:", rand_search.best_params_)


In [ ]:
# Generate predictions with the best model
y_pred = best_rf.predict(X_test)

# Create the confusion matrix
cm = confusion_matrix(y_test, y_pred)

ConfusionMatrixDisplay(confusion_matrix=cm).plot();


In [ ]:
y_pred = best_rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)


In [ ]:
feature_importances = pd.Series(
    best_rf.feature_importances_, index=X_train.columns
).sort_values(ascending=False)

# Plot a simple bar chart
feature_importances.plot.bar();


# Downsampling Approach

status
0    0.75
1    0.25
Name: proportion, dtype: float64

In [29]:
balanced.head()

inspection_id       X_0       X_1       X_2       X_3       X_4       X_5  \
0            448  0.784435  0.000000  0.343279  0.006824  1.561642  0.215440   
1          59807  0.405913  0.000536  0.251625  0.015760  0.773964  0.186458   
2          44580  0.507827  0.000000  0.225031  0.000345  1.274157  0.077392   
3          31650  0.486865  0.000000  0.271573  0.000000  1.320390  0.142131   
4          34813  0.339610  0.000000  0.241342  0.012382  0.913757  0.170984   

   X_6       X_7       X_8       X_9      X_10     X_11      X_12      X_13  \
0  0.0  0.646635  0.237849  0.128509  0.037222  0.00000  0.517970  0.661195   
1  0.0  0.489472  0.295140  0.085539  0.054945  0.00196  0.302255  0.300903   
2  0.0  0.473477  0.090824  0.009540  0.000597  0.00000  0.261105  0.393744   
3  0.0  0.743816  0.241082  0.026052  0.000000  0.00000  0.310518  0.366542   
4  0.0  0.411717  0.176319  0.081881  0.041069  0.00000  0.262840  0.322669   

   X_14      X_15      X_16      X_17  X_18  X_19      X_20  X_21      X_22  \
0   0.0  0.000000  0.228302  0.315093   0.0   0.0  1.152566   0.0  0.760593   
1   0.0  0.001163  0.243965  0.140931   0.0   0.0  0.640572   0.0  0.498696   
2   0.0  0.000000  0.075615  0.120184   0.0   0.0  0.812100   0.0  0.563471   
3   0.0  0.000000  0.265664  0.092204   0.0   0.0  0.832744   0.0  0.605777   
4   0.0  0.000000  0.133847  0.178540   0.0   0.0  0.640512   0.0  0.413106   

       X_23      X_24      X_25      X_26      X_27      X_28      X_29  \
0  0.955436  0.000000  0.091974  3.704097  0.005419  0.001054  0.224377   
1  0.556644  0.000347  0.033947  2.129599  0.068399  0.020146  0.257081   
2  0.720396  0.000000  0.023904  3.115551  0.000000  0.000000  0.047133   
3  0.882115  0.000000  0.002227  3.453398  0.000000  0.017030  0.119031   
4  0.582537  0.000000  0.036566  2.263593  0.003431  0.002227  0.213831   

       X_30      X_31  X_32      X_33      X_34  X_35  X_36      X_37  \
0  0.000000  0.212179   0.0  0.090259  0.201908   0.0   0.0  2.625422   
1  0.000294  0.299840   0.0  0.044748  0.189645   0.0   0.0  1.647569   
2  0.000000  0.045989   0.0  0.012872  0.061871   0.0   0.0  2.296950   
3  0.000000  0.130763   0.0  0.007032  0.128068   0.0   0.0  2.597809   
4  0.000000  0.205942   0.0  0.030734  0.108327   0.0   0.0  1.662861   

       X_38      X_39      X_40  X_41      X_42      X_43      X_44      X_45  \
0  0.000457  0.384522  0.993121   0.0  1.790635  0.143999  2.899524  0.000000   
1  0.005443  0.307750  0.635644   0.0  1.098445  0.076084  1.617025  0.000068   
2  0.000000  0.182387  0.795968   0.0  1.407663  0.040775  2.547612  0.000000   
3  0.000000  0.421130  1.031054   0.0  1.650156  0.010763  2.648007  0.000000   
4  0.001023  0.201459  0.640534   0.0  1.132360  0.088972  1.959414  0.000000   

   X_46  X_47      X_48      X_49  X_50      X_51      X_52  X_53      X_54  \
0   0.0   0.0  0.000000  0.000000   0.0  0.000000  0.036859   0.0  0.157920   
1   0.0   0.0  0.000830  0.000516   0.0  0.001166  0.067987   0.0  0.163022   
2   0.0   0.0  0.000000  0.000000   0.0  0.000000  0.000000   0.0  0.041329   
3   0.0   0.0  0.000000  0.000000   0.0  0.000000  0.001035   0.0  0.059907   
4   0.0   0.0  0.002017  0.000000   0.0  0.000676  0.019675   0.0  0.120829   

   X_55  X_56      X_57  X_58      X_59      X_60  X_61      X_62  X_63  X_64  \
0   0.0   0.0  0.000000   0.0  2.751667  0.000000   0.0  0.432515   0.0   0.0   
1   0.0   0.0  0.002522   0.0  1.650903  0.000168   0.0  0.378210   0.0   0.0   
2   0.0   0.0  0.000000   0.0  2.236040  0.000000   0.0  0.242873   0.0   0.0   
3   0.0   0.0  0.000000   0.0  2.553959  0.000000   0.0  0.324753   0.0   0.0   
4   0.0   0.0  0.000135   0.0  1.785762  0.000000   0.0  0.298919   0.0   0.0   

       X_65      X_66  X_67      X_68      X_69      X_70      X_71      X_72  \
0  0.000000  0.178107   0.0  0.005522  0.004932  0.430323  2.670576  0.366918   
1  0.020439  0.158168   0.0  0.012436  0.042521  0.440580  1.559605  0.